# 03 · M2: метод семантического отклонения и абляция
Три варианта: только эмбеддинг (M2a), только признаки отклонения (M2b), эмбеддинг + признаки (M2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

In [ ]:
from pig.pipeline import load_data, calibrate, task_descriptions
from pig.features.embedder import make_embedder
from pig.models.semantic import SemanticDetector
from pig.eval.metrics import evaluate, summary_table
splits = load_data(cfg)
train, val, test, held = splits['train'], splits['val'], splits['test'], splits['heldout']
emb = make_embedder(cfg, 'sentence', cfg.path('cache'))
sc = cfg['models']['semantic']

In [ ]:
models = {}
for mode in ['emb', 'dev', 'full']:
    m = SemanticDetector(emb, task_descriptions(cfg), mode, sc['C'], sc['window'], sc['stride'], cfg['seed'])
    m.fit(train['text'].tolist(), train['label'], train['attack_type'])
    calibrate(m, val, cfg['threshold']['max_fpr'])
    m.save(cfg.path('models') / f'{m.name}.joblib')
    models[m.name] = m
    print(m.name, round(m.threshold, 4))

## Абляция на test и held-out

In [ ]:
rows = []
for split, part in [('test', test), ('heldout', held)]:
    for name, m in models.items():
        r = evaluate(name, part, m.predict_proba(part['text'].tolist()), m.threshold)
        r['split'] = split
        rows.append(r)
import pandas as pd
abl = pd.DataFrame(rows)[['split', 'model', 'f1', 'f1_ci_low', 'f1_ci_high', 'fpr_benign', 'fpr_hard_neg', 'recall_obfuscation']]
abl.to_csv(cfg.path('reports') / 'ablation.csv', index=False)
abl.round(4)

## Распределения признаков отклонения по классам (train)
Эти графики идут во 2-ю главу: они показывают, что признаки разделяют классы.

In [ ]:
import matplotlib.pyplot as plt
full = models['M2_semantic']
feat = full.deviation_frame(train['text'].tolist())
feat['label'] = train['label'].values
cols = ['d_task', 's_att', 's_ben', 'delta', 'shift', 'drift']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), cols):
    ax.hist([feat.loc[feat.label == 0, c], feat.loc[feat.label == 1, c]], bins=40, label=['норма', 'атака'], alpha=0.8)
    ax.set_title(c); ax.legend()
fig.tight_layout(); fig.savefig(cfg.path('reports') / 'deviation_features.png', dpi=160)
feat.groupby('label')[cols].mean().round(3)

## Атаки, «спрятанные» в нормальный вопрос (обёртка)
Проверка признака `shift`: он должен быть выше у обёрнутых атак.

In [ ]:
wrapped = test[test['template_id'].str.contains('wrap', na=False)]
if len(wrapped):
    f = full.deviation_frame(wrapped['text'].tolist())
    print('shift обёрнутых атак:', f['shift'].mean().round(3), '| shift нормы:', feat.loc[feat.label == 0, 'shift'].mean().round(3))